# PSMAC NPU — ECG Arrhythmia Detection
### MIT-BIH Dataset → MLP → INT8/INT4/INT2 Quantization → NPU Hex Export

**Architecture mapping:**
- Dataset: MIT-BIH Arrhythmia from Kaggle (`shayanfazeli/heartbeat`)
- Features: 187-sample ECG window (one heartbeat)
- Classes: N (Normal), S (Supraventricular), V (Ventricular), F (Fusion), Q (Unknown)
- MLP: 187 → 64 → 32 → 16 → 5  
- Quantized weights exported as hex → loaded into PSMAC weight_mem

In [ ]:
# ── Install / Import ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings, os
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── 1. LOAD DATASET ──────────────────────────────────────────────────────────
# Kaggle dataset: shayanfazeli/heartbeat
# Upload mitbih_train.csv and mitbih_test.csv to /kaggle/input/heartbeat/

train_df = pd.read_csv('/kaggle/input/heartbeat/mitbih_train.csv', header=None)
test_df  = pd.read_csv('/kaggle/input/heartbeat/mitbih_test.csv',  header=None)

CLASS_NAMES = ['N — Normal', 'S — Supraventricular', 'V — Ventricular', 'F — Fusion', 'Q — Unknown']
CLASS_SHORT = ['N', 'S', 'V', 'F', 'Q']
COLORS      = ['#22c55e', '#3b82f6', '#ef4444', '#f59e0b', '#8b5cf6']

X_train = train_df.iloc[:, :187].values.astype(np.float32)
y_train = train_df.iloc[:,  187].values.astype(int)
X_test  = test_df.iloc[:,  :187].values.astype(np.float32)
y_test  = test_df.iloc[:,   187].values.astype(int)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Class distribution (train): {np.bincount(y_train)}')

In [ ]:
# ── 2. DATASET VISUALISATION ─────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10), facecolor='#0f172a')
fig.suptitle('MIT-BIH Arrhythmia — ECG Waveform Gallery', 
             fontsize=20, color='white', fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.5, wspace=0.3)

t = np.linspace(0, 187/360, 187)  # 360 Hz sampling rate → seconds

for cls in range(5):
    idx = np.where(y_train == cls)[0]
    sample = X_train[idx[0]]
    
    ax = fig.add_subplot(gs[0, cls])
    ax.set_facecolor('#1e293b')
    ax.plot(t, sample, color=COLORS[cls], linewidth=1.5)
    ax.fill_between(t, sample, alpha=0.15, color=COLORS[cls])
    ax.set_title(CLASS_SHORT[cls], color=COLORS[cls], fontsize=14, fontweight='bold')
    ax.set_xlabel('Time (s)', color='#94a3b8', fontsize=9)
    ax.tick_params(colors='#64748b', labelsize=8)
    for spine in ax.spines.values(): spine.set_color('#334155')
    ax.grid(True, alpha=0.2, color='#334155')

# Class distribution bar chart
ax_bar = fig.add_subplot(gs[1, :])
ax_bar.set_facecolor('#1e293b')
counts = np.bincount(y_train)
bars = ax_bar.bar(CLASS_SHORT, counts, color=COLORS, alpha=0.85, width=0.5)
for bar, count in zip(bars, counts):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{count:,}', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
ax_bar.set_title('Class Distribution (Training Set)', color='white', fontsize=13, fontweight='bold')
ax_bar.set_ylabel('Sample Count', color='#94a3b8', fontsize=10)
ax_bar.tick_params(colors='#94a3b8')
for spine in ax_bar.spines.values(): spine.set_color('#334155')
ax_bar.set_facecolor('#1e293b')

plt.savefig('ecg_dataset_gallery.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('Saved: ecg_dataset_gallery.png')

In [ ]:
# ── 3. MLP MODEL ─────────────────────────────────────────────────────────────
# Architecture chosen to map EXACTLY onto the 4×4 PSMAC NPU:
#   Layer 1: 187→64  (4 tiles for first 4 cols of output neurons)
#   Layer 2: 64→32
#   Layer 3: 32→16   (16 tiles, one output per tile)
#   Layer 4: 16→5    (final classification head)
#
# Weight quantization targets vec_len=16 (fits in 8-bit addr space)
# with PSMAC mode: INT8(10) | INT4(01) | INT2(00)

class ECG_MLP(nn.Module):
    def __init__(self, input_dim=187, hidden=[64, 32, 16], num_classes=5):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden
        for i in range(len(dims)-1):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.BatchNorm1d(dims[i+1]), nn.ReLU(), nn.Dropout(0.3)]
        layers += [nn.Linear(hidden[-1], num_classes)]
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

model = ECG_MLP().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
# ── 4. TRAINING ───────────────────────────────────────────────────────────────
# Class-weighted loss to handle severe imbalance (N >> V >> F > S > Q)
cw = compute_class_weight('balanced', classes=np.arange(5), y=y_train)
cw_tensor = torch.FloatTensor(cw).to(device)

criterion = nn.CrossEntropyLoss(weight=cw_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

X_tr = torch.FloatTensor(X_train)
y_tr = torch.LongTensor(y_train)
X_te = torch.FloatTensor(X_test)
y_te = torch.LongTensor(y_test)

train_ds = TensorDataset(X_tr, y_tr)
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)

EPOCHS = 40
train_losses, val_accs = [], []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        logits = model(X_te.to(device))
        preds  = logits.argmax(1).cpu().numpy()
        acc    = (preds == y_test).mean() * 100
    
    scheduler.step()
    train_losses.append(total_loss / len(train_dl))
    val_accs.append(acc)
    
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {total_loss/len(train_dl):.4f} | Val Acc: {acc:.2f}%')

print(f'\nFinal Test Accuracy: {val_accs[-1]:.2f}%')

In [ ]:
# ── 5. TRAINING CURVES ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0f172a')
fig.suptitle('Training Dynamics', fontsize=16, color='white', fontweight='bold')

for ax in [ax1, ax2]:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    ax.grid(True, alpha=0.2, color='#334155')
    for spine in ax.spines.values(): spine.set_color('#334155')

ax1.plot(train_losses, color='#f59e0b', linewidth=2)
ax1.fill_between(range(len(train_losses)), train_losses, alpha=0.15, color='#f59e0b')
ax1.set_title('Training Loss', color='white', fontsize=13)
ax1.set_xlabel('Epoch', color='#94a3b8')
ax1.set_ylabel('Cross-Entropy Loss', color='#94a3b8')

ax2.plot(val_accs, color='#22c55e', linewidth=2)
ax2.fill_between(range(len(val_accs)), val_accs, alpha=0.15, color='#22c55e')
ax2.axhline(y=max(val_accs), color='#22c55e', linestyle='--', alpha=0.5)
ax2.text(0, max(val_accs)+0.3, f'Peak: {max(val_accs):.2f}%', color='#22c55e', fontsize=10)
ax2.set_title('Validation Accuracy', color='white', fontsize=13)
ax2.set_xlabel('Epoch', color='#94a3b8')
ax2.set_ylabel('Accuracy (%)', color='#94a3b8')

plt.tight_layout()
plt.savefig('ecg_training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

In [ ]:
# ── 6. CONFUSION MATRIX ───────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    logits_all = model(X_te.to(device))
    probs_all  = torch.softmax(logits_all, dim=1).cpu().numpy()
    preds_all  = logits_all.argmax(1).cpu().numpy()

cm = confusion_matrix(y_test, preds_all)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0f172a')
fig.suptitle('Classification Results — Test Set', fontsize=16, color='white', fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax1,
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            cbar_kws={'label': 'Count'})
ax1.set_title('Confusion Matrix (Raw Counts)', color='white', fontsize=13)
ax1.set_xlabel('Predicted', color='#94a3b8')
ax1.set_ylabel('Actual', color='#94a3b8')
ax1.tick_params(colors='#94a3b8')
ax1.set_facecolor('#1e293b')

# Normalised
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax2,
            xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT,
            vmin=0, vmax=1, cbar_kws={'label': 'Recall'})
ax2.set_title('Normalised Confusion Matrix (Recall)', color='white', fontsize=13)
ax2.set_xlabel('Predicted', color='#94a3b8')
ax2.set_ylabel('Actual', color='#94a3b8')
ax2.tick_params(colors='#94a3b8')

plt.tight_layout()
plt.savefig('ecg_confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\n', classification_report(y_test, preds_all, target_names=CLASS_SHORT))

In [ ]:
# ── 7. QUANTIZATION ENGINE ────────────────────────────────────────────────────
# Maps PSMAC modes:
#   mode 2'b10 → INT8  (8-bit signed, range -128..127)
#   mode 2'b01 → INT4  (4-bit signed, range -8..7)
#   mode 2'b00 → INT2  (2-bit signed, range -2..1)

def quantize_weights(weight_np, bits):
    """Symmetric min-max quantization to `bits` signed integer."""
    n_levels = 2**(bits-1)
    max_val = np.abs(weight_np).max()
    if max_val == 0:
        return np.zeros_like(weight_np, dtype=np.int8)
    scale = max_val / (n_levels - 1)
    q = np.round(weight_np / scale).astype(int)
    q = np.clip(q, -n_levels, n_levels-1)
    return q.astype(np.int8)

def compute_accuracy_quantized(model, X_test_t, y_test_np, bits):
    """Replace Linear weights with quantized versions, eval, restore."""
    original_weights = {}
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            original_weights[name] = module.weight.data.clone()
            q = quantize_weights(module.weight.data.cpu().numpy(), bits)
            module.weight.data = torch.FloatTensor(q).to(device)
    
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t.to(device)).argmax(1).cpu().numpy()
    acc = (preds == y_test_np).mean() * 100
    
    # Restore original FP32 weights
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            module.weight.data = original_weights[name]
    return acc

acc_fp32 = val_accs[-1]
acc_int8 = compute_accuracy_quantized(model, X_te, y_test, 8)
acc_int4 = compute_accuracy_quantized(model, X_te, y_test, 4)
acc_int2 = compute_accuracy_quantized(model, X_te, y_test, 2)

print(f'FP32 Accuracy : {acc_fp32:.2f}%  (baseline, software only)')
print(f'INT8 Accuracy : {acc_int8:.2f}%  (PSMAC mode=2\'b10, {8*1}b storage)')
print(f'INT4 Accuracy : {acc_int4:.2f}%  (PSMAC mode=2\'b01, {4*1}b storage)')
print(f'INT2 Accuracy : {acc_int2:.2f}%  (PSMAC mode=2\'b00, {2*1}b storage)')

In [ ]:
# ── 8. PRECISION-ACCURACY-EFFICIENCY TRADEOFF PLOT ───────────────────────────
# This is the KEY hardware plot — shows what the PSMAC paper is about

modes       = ['FP32\n(Reference)', 'INT8\nmode=10', 'INT4\nmode=01', 'INT2\nmode=00']
accs        = [acc_fp32, acc_int8, acc_int4, acc_int2]
cycles      = [None, 187, 94, 47]   # vec_len / bits_per_byte throughput factor
storage_mb  = [187*64*4/1e6, 187*64*1/1e6, 187*64*0.5/1e6, 187*64*0.25/1e6]  # layer 1 weight mem
speedup     = [1.0, 1.0, 2.0, 4.0]
mode_colors = ['#64748b', '#22c55e', '#3b82f6', '#f59e0b']

fig = plt.figure(figsize=(18, 7), facecolor='#0f172a')
fig.suptitle('PSMAC NPU — Precision vs Accuracy vs Efficiency Trade-off', 
             fontsize=17, color='white', fontweight='bold', y=1.01)
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# --- Plot 1: Accuracy bar ---
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor('#1e293b')
bars = ax1.bar(modes, accs, color=mode_colors, alpha=0.9, width=0.5)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 3,
             f'{acc:.1f}%', ha='center', va='top', color='white', fontsize=11, fontweight='bold')
ax1.set_ylim(min(accs)-5, 102)
ax1.set_title('Classification Accuracy', color='white', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy (%)', color='#94a3b8')
ax1.tick_params(colors='#94a3b8', labelsize=9)
ax1.grid(True, alpha=0.2, color='#334155', axis='y')
for spine in ax1.spines.values(): spine.set_color('#334155')

# --- Plot 2: Cycle count (proportional to compute time) ---
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor('#1e293b')
hw_modes  = modes[1:]  # skip FP32
hw_cycles = cycles[1:]
hw_colors = mode_colors[1:]
bars2 = ax2.bar(hw_modes, hw_cycles, color=hw_colors, alpha=0.9, width=0.5)
for bar, cyc, spdup in zip(bars2, hw_cycles, speedup[1:]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{cyc} cyc\n{spdup:.0f}× speedup', ha='center', va='bottom', 
             color='white', fontsize=10, fontweight='bold')
ax2.set_title('NPU Inference Cycles (Layer 1)', color='white', fontsize=12, fontweight='bold')
ax2.set_ylabel('Cycles per Inference', color='#94a3b8')
ax2.tick_params(colors='#94a3b8', labelsize=9)
ax2.grid(True, alpha=0.2, color='#334155', axis='y')
for spine in ax2.spines.values(): spine.set_color('#334155')

# --- Plot 3: Accuracy vs Speedup scatter ---
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor('#1e293b')
spdup_vals = [1.0, 2.0, 4.0]
acc_vals   = [acc_int8, acc_int4, acc_int2]
labels3    = ['INT8', 'INT4', 'INT2']
for i, (s, a, l, c) in enumerate(zip(spdup_vals, acc_vals, labels3, mode_colors[1:])):
    ax3.scatter(s, a, s=200, color=c, zorder=5)
    ax3.annotate(l, (s, a), textcoords='offset points', xytext=(8, -4),
                 color=c, fontsize=11, fontweight='bold')
ax3.plot(spdup_vals, acc_vals, color='#475569', linewidth=1.5, linestyle='--', zorder=3)
ax3.axhline(y=acc_fp32, color='#64748b', linestyle=':', linewidth=1.5)
ax3.text(1.05, acc_fp32+0.3, 'FP32 baseline', color='#64748b', fontsize=9)
ax3.set_title('Pareto Front: Accuracy vs Speedup', color='white', fontsize=12, fontweight='bold')
ax3.set_xlabel('Inference Speedup (×)', color='#94a3b8')
ax3.set_ylabel('Accuracy (%)', color='#94a3b8')
ax3.tick_params(colors='#94a3b8')
ax3.grid(True, alpha=0.2, color='#334155')
for spine in ax3.spines.values(): spine.set_color('#334155')

plt.savefig('ecg_psmac_tradeoff.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('Saved: ecg_psmac_tradeoff.png')

In [ ]:
# ── 9. LIVE INFERENCE DEMO (single-beat) ──────────────────────────────────────
def npu_inference_demo(ecg_idx, mode_bits):
    """Simulate PSMAC NPU inference for a single ECG beat.
    mode_bits: '10'=INT8, '01'=INT4, '00'=INT2
    """
    mode_map = {'10': (8, 'INT8', '#22c55e'), '01': (4, 'INT4', '#3b82f6'), '00': (2, 'INT2', '#f59e0b')}
    bits, label, color = mode_map[mode_bits]
    
    x = X_test[ecg_idx]
    true_class = y_test[ecg_idx]
    
    acc_q = compute_accuracy_quantized(model, torch.FloatTensor(x.reshape(1,-1)), 
                                        np.array([true_class]), bits)
    
    # Get softmax probs under quantized model
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            q = quantize_weights(module.weight.data.cpu().numpy(), bits)
            module.weight.data = torch.FloatTensor(q).to(device)
    model.eval()
    with torch.no_grad():
        logit = model(torch.FloatTensor(x.reshape(1,-1)).to(device))
        prob  = torch.softmax(logit, dim=1).cpu().numpy()[0]
        pred  = prob.argmax()
    
    cycles_est = {8: 187, 4: 94, 2: 47}[bits]
    vec_len    = 187
    
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor='#0f172a')
    fig.suptitle(f'PSMAC NPU — Live ECG Inference  |  Mode: {label} (mode={mode_bits})', 
                 fontsize=15, color=color, fontweight='bold')
    
    t = np.linspace(0, 187/360, 187)
    # Panel 1: ECG waveform
    ax = axes[0]
    ax.set_facecolor('#1e293b')
    true_color = COLORS[true_class]
    ax.plot(t, x, color=true_color, linewidth=2)
    ax.fill_between(t, x, alpha=0.15, color=true_color)
    ax.set_title(f'Input ECG Beat\nTrue Class: {CLASS_SHORT[true_class]}', color='white', fontsize=12)
    ax.set_xlabel('Time (s)', color='#94a3b8')
    ax.tick_params(colors='#64748b')
    ax.grid(True, alpha=0.2, color='#334155')
    for spine in ax.spines.values(): spine.set_color('#334155')
    ax.text(0.02, 0.95, f'Sample #{ecg_idx}', transform=ax.transAxes,
            color='#64748b', fontsize=9)
    
    # Panel 2: PSMAC output confidences
    ax2 = axes[1]
    ax2.set_facecolor('#1e293b')
    bar_colors = [COLORS[i] if i==pred else '#334155' for i in range(5)]
    bars = ax2.barh(CLASS_SHORT, prob*100, color=bar_colors, alpha=0.9)
    for bar, p in zip(bars, prob):
        ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{p*100:.1f}%', va='center', color='white', fontsize=10)
    ax2.set_xlim(0, 115)
    ax2.set_title(f'NPU Output ({label})\nPrediction: {CLASS_SHORT[pred]} — {"✓ CORRECT" if pred==true_class else "✗ WRONG"}',
                  color='#22c55e' if pred==true_class else '#ef4444', fontsize=12)
    ax2.set_xlabel('Confidence (%)', color='#94a3b8')
    ax2.tick_params(colors='#94a3b8')
    for spine in ax2.spines.values(): spine.set_color('#334155')
    
    # Panel 3: Hardware stats
    ax3 = axes[2]
    ax3.set_facecolor('#1e293b')
    ax3.axis('off')
    stats = [
        ('Precision Mode',   f'{label}  (mode = 2\'b{mode_bits})'),
        ('Vector Length',    f'{vec_len} samples'),
        ('Cycles (est.)',    f'{cycles_est} cycles'),
        ('Speedup vs INT8',  f'{187//cycles_est}× faster' if bits < 8 else 'baseline'),
        ('Weight Storage',   f'{bits * vec_len * 64 // 8} bytes (L1)'),
        ('NPU Tiles Active', '4×4 = 16 tiles'),
        ('Confidence',       f'{prob[pred]*100:.1f}%'),
        ('Decision',         f'{CLASS_NAMES[pred]}'),
    ]
    for i, (k, v) in enumerate(stats):
        y_pos = 0.88 - i * 0.11
        ax3.text(0.05, y_pos, k, transform=ax3.transAxes,
                 color='#64748b', fontsize=10, fontweight='bold')
        ax3.text(0.05, y_pos-0.055, v, transform=ax3.transAxes,
                 color=color, fontsize=11)
    ax3.set_title('Hardware Stats', color='white', fontsize=12)
    
    plt.tight_layout()
    fname = f'ecg_demo_{label}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0f172a')
    plt.show()
    return fname

# Demo: pick a ventricular beat (class 2) for maximum drama
v_indices = np.where(y_test == 2)[0]
demo_idx  = v_indices[5]

npu_inference_demo(demo_idx, '10')  # INT8
npu_inference_demo(demo_idx, '01')  # INT4
npu_inference_demo(demo_idx, '00')  # INT2

In [ ]:
# ── 10. EXPORT QUANTIZED WEIGHTS AS HEX (for PSMAC weight_mem.v) ─────────────
# weight_mem.v accepts: we_w=1, w_addr, w_data (8-bit)
# For INT4: pack two 4-bit weights per byte (high nibble + low nibble)
# For INT2: pack four 2-bit weights per byte

def export_weights_hex(model, layer_idx, bits, filename):
    layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
    W = layers[layer_idx].weight.data.cpu().numpy()
    W_q = quantize_weights(W, bits)
    
    with open(filename, 'w') as f:
        f.write(f'// PSMAC weight_mem hex — Layer {layer_idx} — {bits}-bit quantized\n')
        f.write(f'// Shape: {W.shape} ({W.shape[0]} neurons × {W.shape[1]} inputs)\n')
        f.write(f'// Load with: we_w=1, w_addr=<addr>, w_data=<hex_byte>\n\n')
        f.write('@00\n')  # Verilog $readmemh format
        
        if bits == 8:
            for val in W_q.flatten():
                f.write(f'{val & 0xFF:02X}\n')
        elif bits == 4:
            flat = W_q.flatten()
            for i in range(0, len(flat)-1, 2):
                packed = ((flat[i] & 0xF) << 4) | (flat[i+1] & 0xF)
                f.write(f'{packed & 0xFF:02X}\n')
        elif bits == 2:
            flat = W_q.flatten()
            for i in range(0, len(flat)-3, 4):
                packed = ((flat[i]&3)<<6)|((flat[i+1]&3)<<4)|((flat[i+2]&3)<<2)|(flat[i+3]&3)
                f.write(f'{packed & 0xFF:02X}\n')
    
    print(f'Exported {filename}: {W_q.flatten().shape[0]} weights, {bits}-bit → {filename}')
    return W_q

# Export all 3 modes for layer 1 (187→64)
W8 = export_weights_hex(model, 0, 8, 'weights_int8_layer1.hex')
W4 = export_weights_hex(model, 0, 4, 'weights_int4_layer1.hex')
W2 = export_weights_hex(model, 0, 2, 'weights_int2_layer1.hex')

print('\n✓ Weight files ready for PSMAC loading via $readmemh in weight_mem.v')

In [ ]:
# ── 11. WEIGHT DISTRIBUTION PLOT ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='#0f172a')
fig.suptitle('Weight Distributions After Quantization (Layer 1: 187→64)', 
             fontsize=15, color='white', fontweight='bold')

configs = [(W8, 'INT8 (8-bit)', '#22c55e', 8), 
           (W4, 'INT4 (4-bit)', '#3b82f6', 4), 
           (W2, 'INT2 (2-bit)', '#f59e0b', 2)]

for ax, (W, title, color, bits) in zip(axes, configs):
    ax.set_facecolor('#1e293b')
    unique, counts = np.unique(W.flatten(), return_counts=True)
    ax.bar(unique, counts, color=color, alpha=0.8, width=0.8)
    ax.set_title(title, color=color, fontsize=12, fontweight='bold')
    ax.set_xlabel('Quantized Weight Value', color='#94a3b8')
    ax.set_ylabel('Frequency', color='#94a3b8')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values(): spine.set_color('#334155')
    ax.grid(True, alpha=0.2, color='#334155', axis='y')
    ax.text(0.05, 0.92, f'Range: {unique.min()}..{unique.max()}\nLevels: {len(unique)}',
            transform=ax.transAxes, color='white', fontsize=10,
            bbox=dict(facecolor='#0f172a', alpha=0.7, boxstyle='round'))

plt.tight_layout()
plt.savefig('ecg_weight_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

In [ ]:
# ── 12. SUMMARY TABLE ────────────────────────────────────────────────────────
print('='*65)
print('  PSMAC NPU — ECG Arrhythmia Detection — Final Summary')
print('='*65)
print(f'  Dataset    : MIT-BIH Arrhythmia (shayanfazeli/heartbeat)')
print(f'  Model      : MLP 187→64→32→16→5 ({total_params:,} params)')
print(f'  Hardware   : PSMAC NPU 4×4 tile array (DE2115, Cyclone IV)')
print()
print(f'  {"Mode":<10} {"Bits":<8} {"Accuracy":<14} {"Cycles":<12} {"Speedup":<10} {"Mem (L1)"}')
print(f'  {"-"*64}')
print(f'  {"FP32":<10} {"32":<8} {acc_fp32:.2f}%{"":>6} {"—":<12} {"1.0×":<10} {187*64*4} B')
print(f'  {"INT8":<10} {"8":<8} {acc_int8:.2f}%{"":>6} {"187":<12} {"1.0×":<10} {187*64*1} B')
print(f'  {"INT4":<10} {"4":<8} {acc_int4:.2f}%{"":>6} {"94":<12} {"2.0×":<10} {187*64//2} B')
print(f'  {"INT2":<10} {"2":<8} {acc_int2:.2f}%{"":>6} {"47":<12} {"4.0×":<10} {187*64//4} B')
print('='*65)
print('\nOutput files:')
for f in ['ecg_dataset_gallery.png','ecg_training_curves.png','ecg_confusion_matrix.png',
          'ecg_psmac_tradeoff.png','ecg_weight_distributions.png',
          'weights_int8_layer1.hex','weights_int4_layer1.hex','weights_int2_layer1.hex']:
    print(f'  ✓ {f}')